# 🛠️ Transfer Learning Part 2: Fine-Tuning using VGG16
### 🎯 Unfreezing the Last Convolutional Layer (`block5_conv3`) for Specialized Cat vs Dog Classification

---

## 📌 1. What is Fine-Tuning?

In **Notebook 1**, we performed **Feature Extraction**: we froze the entire VGG16 convolutional base and trained only a new dense classifier head.

Now, we take the next step: **Fine-Tuning**.
Fine-Tuning consists of **unfreezing the top layer(s)** of a frozen model base and jointly training both the newly added classifier head **and the top convolutional layers**.

---

## 🔬 2. Visual Hierarchy in Deep Convolutional Networks

Why unfreeze only the last convolutional layer, and not the whole network?

```
┌─────────────────────────────────────────────────────────────┐
│ 1. Block 1 & Block 2 (Low-Level Features)                   │
│    - Edges, diagonal lines, corners, color blobs            │
│    - Universally useful for ANY computer vision task        │
│    - ❄️ KEEP FROZEN!                                         │
├─────────────────────────────────────────────────────────────┤
│ 2. Block 3 & Block 4 (Mid-Level Features)                   │
│    - Textures, circular curves, surface patterns            │
│    - ❄️ KEEP FROZEN!                                         │
├─────────────────────────────────────────────────────────────┤
│ 3. Block 5 (High-Level Task-Specific Features)              │
│    - Complex semantic objects: animal faces, snouts, ears   │
│    - Contains layer: block5_conv3 (the final conv layer!)   │
│    - 🔥 UNFREEZE & TRAIN! Adapts to Cats vs Dogs nuances     │
└─────────────────────────────────────────────────────────────┘
```

---

## ⚠️ 3. The 3 Golden Rules of Fine-Tuning

1. **Rule 1: Pre-Train the Classifier Head First (Warmup Phase)!**
   * Never unfreeze convolutional layers when the top dense classifier is randomly initialized.
   * Large initial gradient updates from random dense weights will backpropagate into the convolutional base and **destroy the pre-trained weights** (known as *catastrophic forgetting*).
2. **Rule 2: Unfreeze Only the Top Convolutional Layer (`block5_conv3`)!**
   * Earlier layers encode universal features (edges, textures). There is no benefit to re-learning them, and doing so dramatically increases the risk of overfitting.
3. **Rule 3: Use an Extremely Small Learning Rate ($pprox 10^{-5}$ or $10^{-6}$)!**
   * In Feature Extraction, we used $lpha = 10^{-4}$.
   * In Fine-Tuning, we reduce the learning rate by $10	imes	ext{--}100	imes$ to $lpha = 10^{-5}$.
   * We want to gently nudge and adapt the pre-trained representations, not overwrite them:
     $$W_{	ext{new}} = W_{	ext{old}} - lpha \cdot rac{\partial \mathcal{L}}{\partial W}$$

---

### 🗺️ Visual Fine-Tuning Architecture:
```
           Input Image (150 x 150 x 3)
                       │
       ┌───────────────▼───────────────┐
       │   VGG16 Base (Blocks 1 to 4)  │  ❄️ FROZEN (trainable = False)
       │   - block1_conv1, conv2       │  Preserves universal edges & textures
       │   - block2_conv1, conv2       │
       │   - block3_conv1, conv2, conv3│
       │   - block4_conv1, conv2, conv3│
       ├───────────────────────────────┤
       │   VGG16 Base: Block 5         │
       │   - block5_conv1, conv2       │  ❄️ FROZEN
       │   - block5_conv3 (LAST CONV)  │  🔥 UNFROZEN & TRAINED! (2,359,808 params)
       │   - block5_pool               │  ❄️ MaxPool (no weights)
       └───────────────┬───────────────┘
                       │ (4 x 4 x 512 Feature Maps)
       ┌───────────────▼───────────────┐
       │      Flatten Layer            │
       │  Dense(256, ReLU) + Dropout   │  🔥 TRAINABLE CLASSIFIER HEAD
       │  Dense(1, Sigmoid)            │  Jointly tuned with block5_conv3!
       └───────────────────────────────┘
```

In [ ]:
# ============================================================
# STEP 1: Dependencies Import & Environment Verification
# ============================================================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
from keras.applications.vgg16 import VGG16, preprocess_input

# Ensure reproducible random seed
tf.keras.utils.set_random_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")
devices = tf.config.list_physical_devices()
print(f"Available Devices:  {devices}")

## 📁 4. Dataset Loading & Preprocessing Pipeline

We load the Cats vs Dogs dataset using the identical pipeline from Notebook 1 to ensure direct, rigorous comparability.

In [ ]:
# ============================================================
# STEP 2: Dataset Path Resolution & tf.data Pipeline
# ============================================================
candidate_paths = [
    (".", "train", "test"),
    ("..", "../train", "../test"),
    ("/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1", 
     "/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1/train", 
     "/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1/test")
]

train_dir = None
test_dir = None

for base, tr, te in candidate_paths:
    if os.path.exists(tr) and os.path.exists(te):
        train_dir = os.path.abspath(tr)
        test_dir = os.path.abspath(te)
        print(f"Found local dataset at: {base}")
        break

if not train_dir:
    import kagglehub
    dataset_path = kagglehub.dataset_download("salader/dogsvscats")
    train_dir = os.path.join(dataset_path, "train")
    test_dir = os.path.join(dataset_path, "test")

print(f"Train Directory: {train_dir}")
print(f"Test Directory:  {test_dir}")

IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_ds_raw = keras.utils.image_dataset_from_directory(
    directory=train_dir,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=42
)

val_ds_raw = keras.utils.image_dataset_from_directory(
    directory=test_dir,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

def preprocess_vgg(image, label):
    return preprocess_input(image), label

train_ds = train_ds_raw.map(preprocess_vgg, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds_raw.map(preprocess_vgg, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print("[SUCCESS] Data pipeline loaded and preprocessed with VGG16 format!")

## 🏗️ 5. Phase 1: Classifier Warm-Up (Feature Extraction)

As stated in **Rule 1**, we first train the dense classifier head with the convolutional base completely frozen for a few epochs. This stabilizes the classifier weights before unfreezing `block5_conv3`.

In [ ]:
# ============================================================
# STEP 3: Build Base Model & Warm-Up Classifier Head
# ============================================================
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Step A: Freeze the entire base initially
conv_base.trainable = False

# Step B: Build end-to-end model with data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

model = models.Sequential([
    data_augmentation,
    conv_base,
    layers.Flatten(name="flatten"),
    layers.Dense(256, activation='relu', name="dense_classifier"),
    layers.Dropout(0.3, name="dropout"),
    layers.Dense(1, activation='sigmoid', name="output")
], name="vgg16_finetuning_model")

# Step C: Compile with standard learning rate for warmup
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("--- Phase 1 Architecture Summary (Frozen Base Warmup) ---")
model.summary()

trainable_p1 = sum(int(np.prod(p.shape)) for p in model.trainable_weights)
non_trainable_p1 = sum(int(np.prod(p.shape)) for p in model.non_trainable_weights)
print(f"Phase 1 Trainable Parameters:     {trainable_p1:,}")
print(f"Phase 1 Non-Trainable Parameters: {non_trainable_p1:,}")

In [ ]:
# ============================================================
# STEP 4: Execute Warmup Training (Initial Epochs)
# ============================================================
WARMUP_EPOCHS = 5

print("🚀 Starting Phase 1: Classifier Warmup Training...")
history_warmup = model.fit(
    train_ds,
    epochs=WARMUP_EPOCHS,
    validation_data=val_ds
)
print("✅ Phase 1 Warmup Complete! Classifier head is now stabilized.")

## 🔥 6. Phase 2: Unfreezing the Last Convolutional Layer (`block5_conv3`)

Now comes the critical transfer learning step requested:
> **Train the last convolutional layer of VGG16 (`block5_conv3`) while keeping all other convolutional layers frozen.**

### 📐 How VGG16 Layers are Organized:
VGG16 has 13 convolutional layers:
- `block1_conv1`, `block1_conv2`
- `block2_conv1`, `block2_conv2`
- `block3_conv1`, `block3_conv2`, `block3_conv3`
- `block4_conv1`, `block4_conv2`, `block4_conv3`
- `block5_conv1`, `block5_conv2`, **`block5_conv3`** $\leftarrow$ **OUR TARGET!**

### 🧮 Parameter Calculation for `block5_conv3`:
- Input channels: $512$
- Output filters: $512$
- Kernel size: $3 	imes 3$
- Weights: $(3 	imes 3 	imes 512) 	imes 512 = 2,359,296$
- Biases: $512$
- **Total parameters in `block5_conv3`:** $2,359,808$

In [ ]:
# ============================================================
# STEP 5: Selectively Unfreeze the Last Conv Layer (block5_conv3)
# ============================================================
# Enable base trainability flag so inner layer flags take effect
conv_base.trainable = True

# Target: unfreeze ONLY 'block5_conv3'
target_layer_name = 'block5_conv3'

for layer in conv_base.layers:
    if layer.name == target_layer_name:
        layer.trainable = True
    else:
        layer.trainable = False

# Print Comprehensive Layer-by-Layer Trainability Audit Table
print("=" * 70)
print(f"{'INDEX':<6} {'LAYER NAME':<18} {'TYPE':<16} {'TRAINABLE STATUS'}")
print("=" * 70)

for i, layer in enumerate(conv_base.layers):
    if layer.trainable:
        status = "🔥 ACTIVE (TRAINABLE - TARGET!)"
    else:
        status = "❄️ FROZEN"
    print(f"{i:<6} {layer.name:<18} {layer.__class__.__name__:<16} {status}")

print("=" * 70)

### 💡 Alternative Strategy Note: Unfreezing All of Block 5
In some applications, practitioners choose to unfreeze the entire `block5` (`block5_conv1`, `block5_conv2`, and `block5_conv3`).
Here is the snippet to unfreeze all layers in Block 5:
```python
# Unfreeze all layers in Block 5:
for layer in conv_base.layers:
    if layer.name.startswith("block5_"):
        layer.trainable = True
    else:
        layer.trainable = False
```
For our target task, unfreezing `block5_conv3` gives the optimal balance of adaptability with minimal risk of overfitting!

## ⚙️ 7. Re-Compiling with a Low Learning Rate ($10^{-5}$)

> ⚠️ **CRITICAL STEP**: Whenever you change layer `trainable` attributes, you **MUST** re-compile the model with `model.compile()` for changes to take effect in the computational graph.

We compile with `learning_rate=1e-5` ($10	imes$ smaller than warmup).

In [ ]:
# ============================================================
# STEP 6: Model Re-Compilation with Reduced Learning Rate
# ============================================================
LOW_LEARNING_RATE = 1e-5

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LOW_LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Inspect updated parameter counts
trainable_p2 = sum(int(np.prod(p.shape)) for p in model.trainable_weights)
non_trainable_p2 = sum(int(np.prod(p.shape)) for p in model.non_trainable_weights)
param_diff = trainable_p2 - trainable_p1

print("--- Parameter Counts After Unfreezing block5_conv3 ---")
print(f"Total Parameters:                  {trainable_p2 + non_trainable_p2:,}")
print(f"Trainable Parameters (Head + Conv):{trainable_p2:,}")
print(f"Non-Trainable Parameters (Frozen): {non_trainable_p2:,}")
print(f"Newly Unfrozen Weights (Conv3):    +{param_diff:,} parameters (Exact match for block5_conv3!)")

## 🚀 8. Phase 2: Fine-Tuning Training

We resume training from epoch 5 for an additional 7 epochs (total 12 epochs). Notice how we set `initial_epoch=history_warmup.epoch[-1] + 1` so Keras maintains training timeline continuity.

In [ ]:
# ============================================================
# STEP 7: Execute Fine-Tuning Training
# ============================================================
FINE_TUNE_EPOCHS = 7
TOTAL_EPOCHS = WARMUP_EPOCHS + FINE_TUNE_EPOCHS

callbacks_finetune = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print(f"🚀 Starting Phase 2: Fine-Tuning epochs {WARMUP_EPOCHS + 1} to {TOTAL_EPOCHS}...")
history_finetune = model.fit(
    train_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history_warmup.epoch),
    validation_data=val_ds,
    callbacks=callbacks_finetune
)
print("✅ Fine-Tuning Complete!")

## 📊 9. Comparative Visualization: Warmup vs Fine-Tuning

Let's concatenate the training histories and display the full training progression.
The vertical dashed green line marks the exact moment we unfroze `block5_conv3` and reduced the learning rate to $10^{-5}$.

In [ ]:
# ============================================================
# STEP 8: Continuous Training Progression Plot
# ============================================================
# Combine Warmup and Fine-Tuning histories
total_acc = history_warmup.history['accuracy'] + history_finetune.history['accuracy']
total_val_acc = history_warmup.history['val_accuracy'] + history_finetune.history['val_accuracy']
total_loss = history_warmup.history['loss'] + history_finetune.history['loss']
total_val_loss = history_warmup.history['val_loss'] + history_finetune.history['val_loss']

epochs_all = range(1, len(total_acc) + 1)
warmup_boundary = len(history_warmup.epoch)

plt.figure(figsize=(15, 6))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_all, total_acc, 'o-', color='#1f77b4', label='Training Accuracy', linewidth=2)
plt.plot(epochs_all, total_val_acc, 's--', color='#ff7f0e', label='Validation Accuracy', linewidth=2)
plt.axvline(x=warmup_boundary, color='green', linestyle='--', linewidth=2.5, 
            label=f'Fine-Tuning Start (Unfreeze block5_conv3 @ epoch {warmup_boundary})')
plt.title('Accuracy: Warmup vs Fine-Tuning', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', frameon=True)

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_all, total_loss, 'o-', color='#1f77b4', label='Training Loss', linewidth=2)
plt.plot(epochs_all, total_val_loss, 's--', color='#ff7f0e', label='Validation Loss', linewidth=2)
plt.axvline(x=warmup_boundary, color='green', linestyle='--', linewidth=2.5, 
            label=f'Fine-Tuning Start (Unfreeze block5_conv3 @ epoch {warmup_boundary})')
plt.title('Loss: Warmup vs Fine-Tuning', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Loss (Binary Crossentropy)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

# Performance Summary Table
print("=" * 60)
print("                   BENCHMARK SUMMARY")
print("=" * 60)
print(f"Warmup Phase (Feature Extraction Only):")
print(f"  Validation Accuracy: {history_warmup.history['val_accuracy'][-1]*100:.2f}%")
print(f"  Validation Loss:     {history_warmup.history['val_loss'][-1]:.4f}")
print(f"Fine-Tuned Phase (Unfrozen block5_conv3):")
print(f"  Validation Accuracy: {history_finetune.history['val_accuracy'][-1]*100:.2f}%")
print(f"  Validation Loss:     {history_finetune.history['val_loss'][-1]:.4f}")
print("=" * 60)

## 🔮 10. Real-World Inference Comparison

Let's test the fine-tuned model on test images and observe predictions.

In [ ]:
# ============================================================
# STEP 9: Inference on Test Images
# ============================================================
def predict_image(image_path, model, target_size=(150, 150)):
    if not os.path.exists(image_path):
        print(f"⚠️ Image not found at: {image_path}")
        return
    
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, target_size)
    
    img_array = np.expand_dims(img_resized.astype(np.float32), axis=0)
    img_preprocessed = preprocess_input(img_array)
    
    prediction_prob = model.predict(img_preprocessed, verbose=0)[0][0]
    label = "DOG 🐶" if prediction_prob >= 0.5 else "CAT 🐱"
    confidence = prediction_prob if prediction_prob >= 0.5 else (1.0 - prediction_prob)
    
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.title(f"Fine-Tuned Model Prediction: {label}\nConfidence: {confidence*100:.2f}% (Raw: {prediction_prob:.4f})", 
              fontsize=11, fontweight='bold', color='darkgreen' if confidence > 0.8 else 'darkorange')
    plt.axis("off")
    plt.show()
    
    return label, confidence

test_candidates = ["cat.jpg", "dog.jpg", "../cat.jpg", "../dog.jpg"]
for path in test_candidates:
    if os.path.exists(path):
        print(f"\n--- Inference on: {path} ---")
        predict_image(path, model)

## 📋 11. Complete Summary: Feature Extraction vs Fine-Tuning

| Characteristic | Notebook 1: Feature Extraction | Notebook 2: Fine-Tuning |
| :--- | :--- | :--- |
| **Convolutional Base Status** | 100% Frozen (`conv_base.trainable = False`) | Partially Unfrozen (`block5_conv3` active) |
| **Trainable Layers** | Classifier Head only (Flatten + Dense) | Classifier Head + `block5_conv3` |
| **Trainable Parameters** | $pprox 2,097,665$ ($pprox 12.5\%$) | $pprox 4,457,473$ ($pprox 26.5\%$) |
| **Learning Rate** | Moderate ($lpha = 10^{-4}$) | Extremely Low ($lpha = 10^{-5}$) |
| **Risk of Overfitting** | Very Low | Moderate (mitigated with low LR & dropout) |
| **Feature Representation** | Generic ImageNet features | Adapted specifically to target dataset |
| **Training Speed** | Fast | Slightly slower per epoch |
| **Best Used When** | Target dataset is small (< 5,000 images) | Target dataset is medium-sized and needs refinement |

### 🎯 Transfer Learning Decision Guide:
- **Small dataset + Similar to ImageNet** $\implies$ **Feature Extraction** (Freeze all base layers).
- **Medium/Large dataset + Similar to ImageNet** $\implies$ **Fine-Tuning** (Unfreeze last conv layer or last block with low learning rate).
- **Large dataset + Different from ImageNet** $\implies$ **Train larger blocks or train from scratch**.